In [4]:
import os
import time
import glob
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import librosa
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    classification_report
)

# CONFIG
OUTPUT_DIR      = "/mnt/c/Users/Caillou/PycharmProjects/PythonProject"
MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, "deepfake_detector.keras")
NORM_STATS_PATH = os.path.join(OUTPUT_DIR, "norm_stats.npy")
GRAPHS_DIR      = os.path.join(OUTPUT_DIR, "results", "dissertation_graphs")
os.makedirs(GRAPHS_DIR, exist_ok=True)

DATASET_ROOT = "/mnt/c/Users/Caillou/PycharmProjects/PythonProject/dataset/audio"
REAL_DIR     = os.path.join(DATASET_ROOT, "real")
FAKE_DIR     = os.path.join(DATASET_ROOT, "fake")

SAMPLE_RATE  = 16000
DURATION     = 3.0
N_MELS       = 128
HOP_LENGTH   = 512
N_FFT        = 2048
RANDOM_SEED  = 42
TEST_SAMPLES = 2000

# Colors
C_REAL = "#2ECC71"
C_FAKE = "#E74C3C"
C_BLUE = "#3498DB"
C_DARK = "#2C3E50"
DPI    = 180

plt.rcParams.update({
    "figure.facecolor" : "#FAFAFA",
    "axes.facecolor"   : "#FAFAFA",
    "axes.grid"        : True,
    "grid.color"       : "#DDDDDD",
    "font.size"        : 11,
    "text.color"       : "#1A1A1A",
    "axes.labelcolor"  : "#1A1A1A",
    "xtick.color"      : "#1A1A1A",
    "ytick.color"      : "#1A1A1A",
    "axes.titlecolor"  : "#1A1A1A",
    "axes.edgecolor"   : "#CCCCCC",
    "legend.framealpha": 0.9,
    "legend.edgecolor" : "#CCCCCC",
})

# HELPERS

def save(fig, name):
    path = os.path.join(GRAPHS_DIR, name)
    fig.savefig(path, dpi=DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"[Saved] {path}")


def get_audio_files(directory):
    files = []
    for ext in ["*.wav", "*.mp3", "*.flac"]:
        files.extend(glob.glob(os.path.join(directory, "**", ext), recursive=True))
    return files


def extract_mel(filepath):
    try:
        audio, sr = librosa.load(filepath, sr=SAMPLE_RATE, mono=True)
    except Exception:
        return None
    target = int(SAMPLE_RATE * DURATION)
    audio  = np.pad(audio, (0, max(0, target - len(audio))))[:target]
    mel    = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=N_MELS,
                                             n_fft=N_FFT, hop_length=HOP_LENGTH)
    return librosa.power_to_db(mel, ref=np.max)


def build_test_set():
    """Sample a small test set from dataset for graph generation."""
    print("[Data] Sampling test set for graph generation...")
    random.seed(RANDOM_SEED)

    real_files = get_audio_files(REAL_DIR)
    fake_files = get_audio_files(FAKE_DIR)

    n = TEST_SAMPLES // 2
    real_sample = random.sample(real_files, min(n, len(real_files)))
    fake_sample = random.sample(fake_files, min(n, len(fake_files)))

    paths  = real_sample + fake_sample
    labels = [0] * len(real_sample) + [1] * len(fake_sample)

    print(f"  Real: {len(real_sample)} | Fake: {len(fake_sample)}")
    return paths, labels


def extract_features(paths, labels):
    """Extract mel spectrograms and normalise."""
    stats = np.load(NORM_STATS_PATH, allow_pickle=True).item()
    mean, std = stats["mean"], stats["std"]

    X, y = [], []
    total = len(paths)
    for i, (path, label) in enumerate(zip(paths, labels)):
        if i % 200 == 0:
            print(f"  Extracting {i}/{total} ...")
        feat = extract_mel(path)
        if feat is not None:
            X.append(feat)
            y.append(label)

    X = np.array(X)[..., np.newaxis]
    X = (X - mean) / std
    y = np.array(y)
    return X, y, paths[:len(y)]


# GRAPHS

def plot_confusion_matrix(y_test, y_pred):
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Real", "Fake"],
                yticklabels=["Real", "Fake"], ax=ax)
    ax.set_title("Confusion Matrix")
    ax.set_ylabel("True Label")
    ax.set_xlabel("Predicted Label")
    plt.tight_layout()
    save(fig, "01_confusion_matrix.png")


def plot_classification_report(y_test, y_pred):
    report  = classification_report(y_test, y_pred,
                                    target_names=["Real", "Fake"],
                                    output_dict=True)
    metrics   = ["precision", "recall", "f1-score"]
    real_vals = [report["Real"][m] for m in metrics]
    fake_vals = [report["Fake"][m] for m in metrics]

    x, w = np.arange(len(metrics)), 0.35
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(x - w/2, real_vals, w, label="Real", color=C_REAL, alpha=0.85)
    ax.bar(x + w/2, fake_vals, w, label="Fake", color=C_FAKE, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([m.capitalize() for m in metrics])
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Score")
    ax.set_title("Classification Report — Per-Class Metrics")
    ax.legend()
    for i, (r, f) in enumerate(zip(real_vals, fake_vals)):
        ax.text(i - w/2, r + 0.02, f"{r:.3f}", ha="center", fontsize=9)
        ax.text(i + w/2, f + 0.02, f"{f:.3f}", ha="center", fontsize=9)
    save(fig, "02_classification_report.png")


def plot_roc(y_test, y_prob):
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc     = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(fpr, tpr, color=C_BLUE, lw=2,
            label=f"ROC Curve (AUC = {roc_auc:.4f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random Classifier")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve")
    ax.legend(loc="lower right")
    save(fig, "03_roc_curve.png")


def plot_precision_recall_curve(y_test, y_prob):
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(recall, precision, color=C_FAKE, lw=2,
            label=f"PR Curve (AP = {ap:.4f})")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Precision-Recall Curve")
    ax.legend()
    save(fig, "04_precision_recall_curve.png")


def plot_threshold_tuning(y_test, y_prob):
    thresholds = np.linspace(0.01, 0.99, 100)
    precisions, recalls, f1s, accs = [], [], [], []
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        tp = np.sum((y_pred == 1) & (y_test == 1))
        fp = np.sum((y_pred == 1) & (y_test == 0))
        fn = np.sum((y_pred == 0) & (y_test == 1))
        tn = np.sum((y_pred == 0) & (y_test == 0))
        p  = tp / (tp + fp + 1e-9)
        r  = tp / (tp + fn + 1e-9)
        f  = 2 * p * r / (p + r + 1e-9)
        a  = (tp + tn) / len(y_test)
        precisions.append(p); recalls.append(r)
        f1s.append(f);        accs.append(a)

    best_t = thresholds[np.argmax(f1s)]
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(thresholds, precisions, label="Precision", color=C_REAL)
    ax.plot(thresholds, recalls,    label="Recall",    color=C_FAKE)
    ax.plot(thresholds, f1s,        label="F1 Score",  color=C_BLUE, lw=2)
    ax.plot(thresholds, accs,       label="Accuracy",  color=C_DARK, linestyle="--")
    ax.axvline(best_t, color="gray", linestyle=":",
               label=f"Best F1 threshold = {best_t:.2f}")
    ax.set_xlabel("Decision Threshold")
    ax.set_ylabel("Score")
    ax.set_title("Threshold Tuning — Precision / Recall / F1 / Accuracy")
    ax.legend()
    save(fig, "05_threshold_tuning.png")
    print(f"  Best threshold for F1: {best_t:.2f}")


def plot_per_class_bar(y_test, y_pred):
    report  = classification_report(y_test, y_pred,
                                    target_names=["Real", "Fake"],
                                    output_dict=True)
    classes = ["Real", "Fake"]
    f1s     = [report[c]["f1-score"]  for c in classes]
    precs   = [report[c]["precision"] for c in classes]
    recs    = [report[c]["recall"]    for c in classes]

    x, w = np.arange(len(classes)), 0.25
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.bar(x - w,   precs, w, label="Precision", color=C_BLUE, alpha=0.85)
    ax.bar(x,       recs,  w, label="Recall",    color=C_REAL, alpha=0.85)
    ax.bar(x + w,   f1s,   w, label="F1 Score",  color=C_FAKE, alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(classes)
    ax.set_ylim(0, 1.1); ax.set_ylabel("Score")
    ax.set_title("Per-Class Performance (Test Set)")
    ax.legend()
    save(fig, "06_per_class_bar.png")


def plot_dataset_distribution():
    n_real = len(get_audio_files(REAL_DIR))
    n_fake = len(get_audio_files(FAKE_DIR))
    fig, ax = plt.subplots(figsize=(6, 6))
    sizes  = [n_real, n_fake]
    labels = [f"Real\n({n_real:,})", f"Fake\n({n_fake:,})"]
    wedges, texts, autotexts = ax.pie(
        sizes, labels=labels, colors=[C_REAL, C_FAKE],
        autopct="%1.1f%%", startangle=140,
        wedgeprops=dict(edgecolor="white", linewidth=2)
    )
    for at in autotexts:
        at.set_fontsize(12); at.set_fontweight("bold")
    ax.set_title("Dataset Distribution — Real vs Fake")
    save(fig, "07_dataset_distribution.png")


def plot_fake_source_breakdown():
    fake_subdirs = [d for d in os.listdir(FAKE_DIR)
                    if os.path.isdir(os.path.join(FAKE_DIR, d))]
    counts = {}
    for subdir in fake_subdirs:
        files = get_audio_files(os.path.join(FAKE_DIR, subdir))
        if len(files) > 0:
            counts[subdir] = len(files)

    counts = dict(sorted(counts.items(), key=lambda x: x[1], reverse=True))
    labels = [k.replace("ASVspoof2019_LA_", "ASVspoof_")
               .replace("ljspeech_", "").replace("LJSpeech-", "")
              for k in counts.keys()]
    values = list(counts.values())

    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.barh(labels, values, color=C_FAKE, alpha=0.8, edgecolor="white")
    ax.set_xlabel("Number of Files")
    ax.set_title("Fake Audio Sources — File Count per Subfolder")
    for bar, val in zip(bars, values):
        ax.text(val + 100, bar.get_y() + bar.get_height()/2,
                f"{val:,}", va="center", fontsize=9)
    ax.invert_yaxis()
    save(fig, "08_fake_source_breakdown.png")


def plot_inference_time(model, test_paths):
    stats = np.load(NORM_STATS_PATH, allow_pickle=True).item()
    mean, std = stats["mean"], stats["std"]

    times = []
    for path in test_paths[:50]:
        feat = extract_mel(path)
        if feat is None:
            continue
        X = (feat[np.newaxis, ..., np.newaxis] - mean) / std
        start = time.time()
        model.predict(X, verbose=0)
        times.append((time.time() - start) * 1000)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(times, bins=20, color=C_BLUE, alpha=0.8, edgecolor="white")
    axes[0].axvline(np.mean(times), color=C_FAKE, linestyle="--",
                    label=f"Mean: {np.mean(times):.1f}ms")
    axes[0].set_xlabel("Inference Time (ms)")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Inference Time Distribution")
    axes[0].legend()

    axes[1].plot(range(len(times)), times, color=C_BLUE, alpha=0.7, linewidth=0.8)
    axes[1].axhline(np.mean(times), color=C_FAKE, linestyle="--",
                    label=f"Mean: {np.mean(times):.1f}ms")
    axes[1].set_xlabel("Sample Index")
    axes[1].set_ylabel("Inference Time (ms)")
    axes[1].set_title("Inference Time per Sample")
    axes[1].legend()
    plt.suptitle(f"Inference Time Analysis (n={len(times)} samples)", fontsize=13)
    plt.tight_layout()
    save(fig, "09_inference_time.png")
    print(f"  Mean: {np.mean(times):.2f}ms | Max: {np.max(times):.2f}ms | Min: {np.min(times):.2f}ms")


# EXCEL EXPORT

def export_to_excel(y_test, y_pred, y_prob, test_paths):
    try:
        import pandas as pd
    except ImportError:
        print("[WARN] pandas not installed — skipping Excel export")
        return

    excel_path = os.path.join(GRAPHS_DIR, "dissertation_metrics.xlsx")
    writer = pd.ExcelWriter(excel_path, engine="openpyxl")

    # Sheet 1 — Classification Report
    report = classification_report(y_test, y_pred,
                                   target_names=["Real", "Fake"],
                                   output_dict=True)
    df_report = pd.DataFrame(report).transpose().round(4)
    df_report.to_excel(writer, sheet_name="Classification Report")

    # Sheet 2 — Per-file predictions
    df_preds = pd.DataFrame({
        "filepath"        : test_paths,
        "true_label"      : ["Real" if l == 0 else "Fake" for l in y_test],
        "predicted_label" : ["Real" if p == 0 else "Fake" for p in y_pred],
        "confidence_fake" : np.round(y_prob, 4),
        "confidence_real" : np.round(1 - y_prob, 4),
        "correct"         : (y_pred == y_test),
    })
    df_preds.to_excel(writer, sheet_name="Predictions", index=False)

    # Sheet 3 — Summary metrics
    tp = int(np.sum((y_pred == 1) & (y_test == 1)))
    tn = int(np.sum((y_pred == 0) & (y_test == 0)))
    fp = int(np.sum((y_pred == 1) & (y_test == 0)))
    fn = int(np.sum((y_pred == 0) & (y_test == 1)))
    df_summary = pd.DataFrame({
        "Metric": ["Accuracy", "Precision", "Recall", "F1 Score",
                   "True Positives", "True Negatives",
                   "False Positives", "False Negatives",
                   "Total Samples"],
        "Value": [
            round((tp + tn) / len(y_test), 4),
            round(tp / (tp + fp + 1e-9), 4),
            round(tp / (tp + fn + 1e-9), 4),
            round(2 * tp / (2 * tp + fp + fn + 1e-9), 4),
            tp, tn, fp, fn, len(y_test)
        ]
    })
    df_summary.to_excel(writer, sheet_name="Summary Metrics", index=False)

    # Sheet 4 — Fake source breakdown
    fake_subdirs = [d for d in os.listdir(FAKE_DIR)
                    if os.path.isdir(os.path.join(FAKE_DIR, d))]
    source_data = []
    for subdir in fake_subdirs:
        files = get_audio_files(os.path.join(FAKE_DIR, subdir))
        if len(files) > 0:
            source_data.append({"Source": subdir, "File Count": len(files)})
    df_sources = pd.DataFrame(source_data).sort_values("File Count", ascending=False)
    df_sources.to_excel(writer, sheet_name="Fake Sources", index=False)

    writer.close()
    print(f"[Saved] Excel → {excel_path}")


# MAIN

def main():
    print("=" * 50)
    print("  Dissertation Graph Generator (Standalone)")
    print("=" * 50)

    # Load model
    print("\n[Load] Loading model...")
    model = tf.keras.models.load_model(MODEL_SAVE_PATH)
    print(f"[Load] Model loaded from {MODEL_SAVE_PATH}")

    # Build test set
    paths, labels = build_test_set()

    # Extract features
    print("\n[Features] Extracting features...")
    X_test, y_test, test_paths = extract_features(paths, labels)
    print(f"[Features] Done — shape: {X_test.shape}")

    # Predict
    print("\n[Predict] Running predictions...")
    y_prob = model.predict(X_test, verbose=1).flatten()
    y_pred = (y_prob >= 0.5).astype(int)

    # Print quick summary
    print("\n=== Quick Summary ===")
    print(classification_report(y_test, y_pred, target_names=["Real", "Fake"]))

    # Generate all graphs
    print("\n[Graphs] Generating dissertation graphs...")
    plot_confusion_matrix(y_test, y_pred)
    plot_classification_report(y_test, y_pred)
    plot_roc(y_test, y_prob)
    plot_precision_recall_curve(y_test, y_prob)
    plot_threshold_tuning(y_test, y_prob)
    plot_per_class_bar(y_test, y_pred)
    plot_dataset_distribution()
    plot_fake_source_breakdown()
    plot_inference_time(model, test_paths)

    # Export to Excel
    print("\n[Excel] Exporting metrics to Excel...")
    export_to_excel(y_test, y_pred, y_prob, test_paths)

    print(f"\n[Done] All graphs saved to : {GRAPHS_DIR}")
    print(f"[Done] Excel saved to      : {GRAPHS_DIR}/dissertation_metrics.xlsx")
    print(f"[Done] Total: 9 graphs + 1 Excel file generated")


if __name__ == "__main__":
    main()

  Dissertation Graph Generator (Standalone)

[Load] Loading model...
[Load] Model loaded from /mnt/c/Users/Caillou/PycharmProjects/PythonProject/deepfake_detector.keras
[Data] Sampling test set for graph generation...
  Real: 1000 | Fake: 1000

[Features] Extracting features...
  Extracting 0/2000 ...
  Extracting 200/2000 ...
  Extracting 400/2000 ...
  Extracting 600/2000 ...
  Extracting 800/2000 ...
  Extracting 1000/2000 ...
  Extracting 1200/2000 ...
  Extracting 1400/2000 ...
  Extracting 1600/2000 ...
  Extracting 1800/2000 ...
[Features] Done — shape: (2000, 128, 94, 1)

[Predict] Running predictions...
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step

=== Quick Summary ===
              precision    recall  f1-score   support

        Real       0.98      0.99      0.99      1000
        Fake       0.99      0.98      0.99      1000

    accuracy                           0.99      2000
   macro avg       0.99      0.99      0.99      2000
weighted avg       0.99      0.99      0.99  